# AI4I condition classifier: local / Workbench

Educational synthetic data, no future failure horizon. This notebook reuses the same modules, split seed and selection procedure as the local application. No cloud resources or BigQuery jobs are created by these cells. Use Python 3.11. Install requirements-dev.txt in that kernel before running. In Workbench, upload the project without credentials or .venv and select the project root below.

In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidates if (p / 'src' / 'train.py').exists()), None)
if project_root is None:
    raise RuntimeError('Open this notebook inside the uploaded project/cloud directory.')
os.chdir(project_root)
sys.path.insert(0, str(project_root))
print('Project:', project_root, '\nPython:', sys.version)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError('Use a Python 3.11 kernel for the locked experiment.')

Project: YOUR_PROJECT_DIRECTORY 
Python: 3.11.16 (main, Sep  1 2026, 14:15:24) [MSC v.1944 64 bit (AMD64)]


In [2]:
from src.data_loader import download_dataset, load_dataset, dataset_report
from src.config import DATA_PATH
if not DATA_PATH.exists():
    download_dataset()
dataset = load_dataset()
report = dataset_report(dataset)
print({key: report[key] for key in ['rows', 'target_counts', 'label_inconsistencies']})
dataset.head()

{'rows': 10000, 'target_counts': {'0': 9661, '1': 339}, 'label_inconsistencies': 27}


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Preprocessing, fitting and validation-only selection

The same module creates a 60/20/20 stratified split, scaled/OHE LR and OHE RF, three-fold training-internal calibration candidates, validation-selected thresholds and the frozen model selection before test evaluation. Existing artifacts are never silently retrained. For a new experiment, archive models/ and artifacts/ before starting.

In [3]:
from src.train import split_dataset
from src.preprocessing import model_candidates
splits = split_dataset(dataset)
print({name: {'rows': len(part), 'failures': int(part['Machine failure'].sum())} for name, part in splits.items()})
model_candidates()

{'train': {'rows': 6000, 'failures': 203}, 'validation': {'rows': 2000, 'failures': 68}, 'test': {'rows': 2000, 'failures': 68}}


{'Majority baseline': DummyClassifier(strategy='most_frequent'),
 'Calibrated Logistic Regression': CalibratedClassifierCV(cv=3,
                        estimator=Pipeline(steps=[('pre',
                                                   ColumnTransformer(transformers=[('numeric',
                                                                                    StandardScaler(),
                                                                                    ['Air '
                                                                                     'temperature '
                                                                                     '[K]',
                                                                                     'Process '
                                                                                     'temperature '
                                                                                     '[K]',
                                            

In [4]:
import subprocess
from src.config import MODELS
if not (MODELS / 'metadata.json').exists():
    subprocess.run([sys.executable, '-m', 'src.train'], check=True)
else:
    print('Loading the completed original experiment; no retraining performed.')
if not (MODELS / 'probability_metadata.json').exists():
    subprocess.run([sys.executable, '-m', 'src.probability_profile'], check=True)
else:
    print('Loading the persisted calibrated probability profile.')

Loading the completed original experiment; no retraining performed.
Loading the persisted calibrated probability profile.


In [5]:
import json
import pandas as pd
from src.config import ARTIFACTS
results = json.loads((ARTIFACTS / 'evaluation.json').read_text())
metrics = ['accuracy', 'precision', 'recall', 'f1', 'average_precision', 'roc_auc', 'brier_score']
comparison = pd.DataFrame({name: {key: result['test'][key] for key in metrics} for name, result in results.items()}).T
comparison

,accuracy,precision,recall,f1,average_precision,roc_auc,brier_score
Majority baseline,0.9660,0.000000,0.000000,0.000000,0.034000,0.500000,0.034000
Calibrated Logistic Regression,0.9685,0.857143,0.088235,0.160000,0.392901,0.903392,0.025639
Random Forest,0.9775,0.925926,0.367647,0.526316,0.742627,0.960841,0.016824
Calibrated Random Forest,0.9730,0.888889,0.235294,0.372093,0.719976,0.957922,0.016521


In [6]:
from src.inference import Predictor
from src.config import FEATURES
predictor = Predictor.load()
print(predictor.metadata['selected_model'], predictor.metadata['threshold'], predictor.metadata['score_kind'])
print('Classification policies:', predictor.metadata['policies'])
predictor.predict(dataset[FEATURES].head())

Calibrated Random Forest 0.8493198346514657 calibrated probability estimate
Classification policies: {'high_precision': {'threshold': 0.8493198346514657, 'method': 'maximum recall at validation precision >= 0.90', 'precision': 0.9, 'recall': 0.2647058823529412, 'precision_target_achieved': True, 'predicted_positives': 20}, 'balanced': {'threshold': 0.42169382344542883, 'method': 'maximum validation F1; no 90% precision constraint', 'precision': 0.7454545454545455, 'recall': 0.6029411764705882, 'f1': 0.6666666666666666}}


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],failure_probability,classification_policy,decision_threshold,elevated_risk,risk_category,outside_training_range
0,M,298.1,308.6,1551,42.8,0,0.007967,high_precision,0.84932,False,Lower,False
1,L,298.2,308.7,1408,46.3,3,0.008216,high_precision,0.84932,False,Lower,False
2,L,298.1,308.5,1498,49.4,5,0.008235,high_precision,0.84932,False,Lower,False
3,L,298.2,308.6,1433,39.5,7,0.007967,high_precision,0.84932,False,Lower,False
4,L,298.2,308.7,1408,40.0,9,0.008047,high_precision,0.84932,False,Lower,False


In [7]:
subprocess.run([sys.executable, 'scripts/export_reports.py'], check=True)
print('Exported artifacts/test_metrics.csv, artifacts/figures/, docs/evaluation_report.md')
print('Model, schema and metadata:', MODELS)

Exported artifacts/test_metrics.csv, artifacts/figures/, docs/evaluation_report.md
Model, schema and metadata: YOUR_PROJECT_DIRECTORY\models


## Interpretation and cleanup

Test results belong to the frozen local methodology, not the reference experiment. The original winner is uncalibrated RF; the dashboard uses the separate calibrated RF probability profile. Both policy thresholds use validation evidence, and profile metrics reuse the previously inspected test partition. Small failure counts, synthetic data and missing external validation limit interpretation. Cloud execution remains unverified until actually run in Workbench. Close/delete any paid notebook resources and inspect retained disks after the assessment.